# MCU JSON Command Set
In the previous chapter, we introduced a simple example. In this example, we send motion control commands from the host computer to the microcontroller. The microcontroller can receive a large number of commands, and in this chapter, we will introduce these commands.

## Composition of a JSON Command
Taking the command we sent in the previous chapter, {"T":1,"L":0.2,"R":0.2}, as an example, in this JSON data, the value T represents the type of the command (Type), the value L represents the target linear velocity of the left (LEFT) wheel, and the value R represents the target linear velocity of the right (RIGHT) wheel. The unit of linear velocity is assumed to be m/s. In summary, this is a motion control command, with the motion control parameters being the target linear velocities of the left and right wheels.

All subsequent JSON commands will include a T value to define the type of the command, but the specific command parameters will vary depending on the type of command.

## JSON Command Set
You can view the definitions of these commands in the json_cmd.h file of our open-source lower-level control examples, or add new lower-level functions yourself.
### Motion Control Commands
These commands are the most basic commands for mobile robots and are used for motion-related function control.

In the following commands, each command contains three parts: example, brief introduction, and detailed introduction.

#### CMD_SPEED_CTRL
- {"T":1,"L":0.5,"R":0.5}
- Set target linear speed for both wheels (closed-loop speed control)
> L and R represent the target linear speeds for the left and right wheels, respectively, in m/s. Negative values represent reverse rotation, and 0 means stop. The range of target linear speeds depends on the motor/gearbox/wheel diameter used in the product. The related formulas can be found in the open-source lower-level control examples. It should be noted that for chassis using brushed DC motors, when the absolute value of the target speed is very small (but not 0), due to the typically poor low-speed performance of brushed DC motors, the product's speed may fluctuate significantly during movement.
#### CMD_PWM_INPUT
- {"T":11,"L":164,"R":164}
- Set PWM values for both drive wheels (open-loop speed control)
> L and R represent the PWM values for the left and right wheels, respectively. The range is -255 to 255, with negative values representing reverse. When the absolute value is 255, the PWM is 100%, indicating full power for that wheel.
#### CMD_ROS_CTRL
- {"T":13,"X":0.1,"Z":0.3}
- ROS control (closed-loop speed control)
> This command is used for controlling the chassis via a ROS upper-level system. X represents the linear velocity, in m/s, and can be negative; Z represents the angular velocity, in rad/s, and can also be negative.
#### CMD_SET_MOTOR_PID
- {"T":2,"P":20,"I":2000,"D":0,"L":255}
- PID controller settings
> This command is used to adjust PID parameters. The PID parameters in the above JSON example are the default parameters for this product. L represents WINDUP_LIMITS, which is a reserved interface that is not currently used in the product.

### OLED Display Control Commands
The product is equipped with an OLED display that communicates with the lower-level ESP32 module via I2C. The upper-level system can send JSON commands to change the content displayed on the screen.
#### CMD_OLED_CTRL
- {"T":3,"lineNum":0,"Text":"putYourTextHere"}
- Control the display screen to show custom content
> lineNum is the line number; a single JSON command can change the content of one line. When the lower-level device receives a new command, the default OLED interface shown at startup will disappear, replaced by the new content you added. For the 0.91-inch OLED screens used in most products, lineNum can be 0, 1, 2, or 3, for a total of four lines. Text is the text you want to display on that line. If the content on this line is too long, it will automatically wrap, but it will also push out the last line.
#### CMD_OLED_DEFAULT
- {"T":-3}
- Control the display screen to show custom content
> Use this command to make the OLED display the default startup screen.

### Module Type
Different types of modules (none/robot arm/gimbal) can be installed on the mobile chassis. Use this command to inform the lower-level device of the currently installed module type. This command is usually sent automatically from the upper-level device when it powers on. Later chapters will cover this part in detail.
#### CMD_MODULE_TYPE
- {"T":4,"cmd":0}
- Set module type
> The value of cmd represents the module type. Currently, there are three types: 0: nothing installed, 1: robot arm, 2: gimbal.

### IMU Related Functions
The chassis is equipped with an IMU sensor. You can use the following commands to obtain IMU data. Note that after powering on the product, continuous chassis information feedback (which includes IMU data) is enabled by default. The IMU-related functions here are only necessary when the continuous chassis information feedback is turned off.
#### CMD_GET_IMU_DATA
- {"T":126}
- Get IMU data
> Sending this command will retrieve IMU data.
#### CMD_CALI_IMU_STEP
- {"T":127}
- IMU calibration (reserved interface)
> The current product firmware does not require calibration; this command is reserved.
#### CMD_GET_IMU_OFFSET
- {"T":128}
- Get current IMU offset (reserved interface)
> Use this command to return the current offset values for each axis of the IMU.

#### CMD_SET_IMU_OFFSET
- {"T":129,"x":-12,"y":0,"z":0}
- Set IMU offset (reserved interface)
> Use this command to set the offset for each axis of the IMU. This command is reserved; the current product does not require using it.
### Chassis Information Feedback

#### CMD_BASE_FEEDBACK
- {"T":130}
- Chassis information feedback
> After the product is powered on, chassis information feedback is usually enabled by default and is automatic. If the continuous chassis information feedback function is turned off and you need to obtain chassis information once, you can use this command to get basic chassis information.

#### CMD_BASE_FEEDBACK_FLOW
- {"T":131,"cmd":1}
- Continuous chassis information feedback
> Set the value of cmd to 1 to enable this function. This function is enabled by default and will continuously feedback chassis information; when the value of cmd is set to 0, the function is disabled. After disabling, the host can obtain chassis information through CMD_BASE_FEEDBACK.

#### CMD_FEEDBACK_FLOW_INTERVAL
- {"T":142,"cmd":0}
- Set the extra interval time for continuous feedback
> The value of cmd is the extra interval time to set, in milliseconds. This command allows you to adjust the frequency of chassis feedback information.

#### CMD_UART_ECHO_MODE
- {"T":143,"cmd":0}
- Set the command echo mode
> When cmd is set to 0, echo is turned off; when cmd is set to 1, echo is enabled. After enabling command echo mode, the lower device will output the received command.

### WIFI Related Settings

#### CMD_WIFI_ON_BOOT
- {"T":401,"cmd":3}
- Set the WIFI mode at boot.
> When cmd is 0, WIFI is disabled; 1 - AP; 2 - STA; 3 - AP STA.

#### CMD_SET_AP
- {"T":402,"ssid":"UGV","password":"12345678"}
- Set the name and password for AP mode. (ESP32 creates a hotspot)

#### CMD_SET_STA
- {"T":403,"ssid":"WIFI_NAME","password":"WIFI_PASSWORD"}
- Set the name and password for STA mode. (ESP32 connects to a known hotspot)

#### CMD_WIFI_APSTA
- {"T":404,"ap_ssid":"UGV","ap_password":"12345678","sta_ssid":"WIFI_NAME","sta_password":"WIFI_PASSWORD"}
- Set the names and passwords for both AP and STA modes. (AP STA mode)

#### CMD_WIFI_INFO
- {"T":405}
- Get the current WIFI information.

#### CMD_WIFI_CONFIG_CREATE_BY_STATUS
- {"T":406}
- Create a new WIFI configuration file using the current settings.

#### CMD_WIFI_CONFIG_CREATE_BY_INPUT
- {"T":407,"mode":3,"ap_ssid":"UGV","ap_password":"12345678","sta_ssid":"WIFI_NAME","sta_password":"WIFI_PASSWORD"}
- Create a new WIFI configuration file using the input settings.

#### CMD_WIFI_STOP
- {"T":408}
- Disconnect the WIFI connection.

### 12V Switch and Gimbal Settings

#### CMD_LED_CTRL
- {"T":132,"IO4":255,"IO5":255}
- 12V switch output settings
> The product's lower-level device has two 12V switch interfaces, with two ports each for a total of four interfaces. You can use this command to set the output voltage of these interfaces. When the value is 255, it corresponds to the 3S battery voltage.
> By default, the product uses these interfaces to control LED lights. You can use this command to adjust the LED brightness.

#### CMD_GIMBAL_CTRL_SIMPLE
- {"T":133,"X":0,"Y":0,"SPD":0,"ACC":0}
- Basic gimbal control command
> This command is used to control the direction of the gimbal.
> X is the horizontal direction, in degrees; positive to the right, negative to the left, with a range of -180 to 180.
> Y is the vertical direction, in degrees; positive upward, negative downward, with a range of -30 to 90.
> SPD is speed, ACC is acceleration. When set to 0, it means maximum speed/acceleration.

#### CMD_GIMBAL_CTRL_MOVE
- {"T":134,"X":45,"Y":45,"SX":300,"SY":300}
- Continuous gimbal control command
> This command is used for continuous control of the gimbal's direction.
> X is the horizontal direction, in degrees; positive to the right, negative to the left, with a range of -180 to 180.
> Y is the vertical direction, in degrees; positive upward, negative downward, with a range of -30 to 90.
> SX and SY are the speeds for the X-axis and Y-axis respectively.

#### CMD_GIMBAL_CTRL_STOP
- {"T":135}
- Gimbal stop command
> After using the commands above to move the gimbal, you can use this command to make the gimbal stop at any time.

#### CMD_GIMBAL_STEADY
- {"T":137,"s":0,"y":0}
- Gimbal stabilization function
> When s is 0, this function is disabled; when s is 1, it is enabled. When enabled, the gimbal will automatically adjust its vertical angle based on IMU data. Y is the desired angle of the gimbal relative to the ground (even when stabilization is on, the camera can still look up and down).

#### CMD_GIMBAL_USER_CTRL
- {"T":141,"X":0,"Y":0,"SPD":300}
- Gimbal UI control
> This command is used to control the gimbal via the UI. X can be -1, 0, or 1; -1 rotates left, 0 stops, 1 rotates right.
> Y can be -1, 0, or 1; -1 tilts downward, 0 stops, 1 tilts upward.
> SPD stands for speed

### M2 Robotic Arm Control

#### CMD_MOVE_INIT
- {"T":100}
- Moves the robotic arm to its initial posture
> Under normal circumstances, the robotic arm will automatically move to the initial position when powered on. This command will cause a process block.

#### CMD_SINGLE_JOINT_CTRL
- {"T":101,"joint":0,"rad":0,"spd":0,"acc":10}
- Single-joint motion control for the robotic arm
> joint: Joint number.

1: BASE_JOINT (base joint).

2: SHOULDER_JOINT (shoulder joint).

3: ELBOW_JOINT (elbow joint).

4: EOAT_JOINT (wrist/gripper joint).

rad: The target angle to rotate to (in radians), based on each joint's initial position. The default angles and rotation directions for each joint are as follows:

The initial position of the BASE_JOINT is 0, with a rotation range from 3.14 to -3.14. When the angle increases, the base joint turns left; when the angle decreases, the base joint turns right.

The initial position of the SHOULDER_JOINT is 0, with a rotation range from 1.57 to -1.57. When the angle increases, the shoulder joint moves forward; when it decreases, the shoulder joint moves backward.

The initial position of the ELBOW_JOINT is 1.570796, with a rotation range from 3.14 to -1.11. When the angle increases, the elbow joint moves downward; when it decreases, it moves in the opposite direction.

The initial position of the EOAT_JOINT is 3.141593. By default, this is the gripper joint, with a rotation range from 1.08 to 3.14. When the angle decreases, the gripper opens.

If changed to a wrist joint, the rotation range is 1.08 to 5.20. When the angle increases, the wrist moves downward; when it decreases, the wrist moves upward.

spd: Rotation speed, in steps per second. The servo completes one revolution in 4096 steps. Higher values increase speed. A value of 0 rotates at maximum speed.

acc: Acceleration at the start and end of rotation. Smaller values make motion smoother. The value can range from 0-254, with units of 100 steps/second². For example, setting it to 10 results in acceleration of 1000 steps per second². A value of 0 uses maximum acceleration.

#### CMD_JOINTS_RAD_CTRL
- {"T":102,"base":0,"shoulder":0,"elbow":1.57,"hand":1.57,"spd":0,"acc":10}
- Control the rotation of all robot arm joints in radians  
> base: Angle of the base joint. The rotation range can be found in the description of the 'rad' key in the "CMD_SINGLE_JOINT_CTRL" command above.  

shoulder: Angle of the shoulder joint.  

elbow: Angle of the elbow joint.  

hand: Angle of the gripper/wrist joint.  

spd: Rotation speed, unit is steps/second. One full rotation of the servo is 4096 steps. The higher the value, the faster the speed. When the speed value is 0, it rotates at maximum speed.  

acc: Acceleration at the start and end of rotation. The smaller the value, the smoother the start and stop. The value can be 0-254, unit is 100 steps/second². For example, if set to 10, the speed changes with an acceleration of 1000 steps per second squared. When the acceleration value is 0, it rotates with maximum acceleration.  

#### CMD_SINGLE_AXIS_CTRL  
- {"T":103,"axis":2,"pos":0,"spd":0.25}  
- Single-axis coordinate control  
> axis: 1-x, 2-y, 3-z, 4-t. For axes other than T, the pos parameter is in mm. For T axis, the unit is radians. spd is the speed coefficient; higher values mean faster speed.  

#### CMD_XYZT_GOAL_CTRL  
- {"T":104,"x":235,"y":0,"z":234,"t":3.14,"spd":0.25}  
- Robot arm coordinate motion control (inverse kinematics)  
> This function will cause blocking  

#### CMD_XYZT_DIRECT_CTRL  
- {"T":1041,"x":235,"y":0,"z":234,"t":3.14}  
- Robot arm coordinate motion control (inverse kinematics)  
> This function will not cause blocking  

#### CMD_SERVO_RAD_FEEDBACK  
- {"T":105}  
- Feedback of the robot arm’s coordinate information  

#### CMD_EOAT_HAND_CTRL  
- {"T":106,"cmd":1.57,"spd":0,"acc":0}  
- End-effector joint control (radians)  
> cmd: The target rotation angle in radians. The initial position of the EOAT_JOINT is 3.141593 by default.  

The default product is the gripper joint, with an angle rotation range of 1.08 to 3.14. When the angle decreases, the gripper opens.  

If replaced with the wrist joint, the angle rotation range is 1.08 to 5.20. When the angle increases, the wrist rotates downward; when the angle decreases, the wrist rotates upward.  

spd: Rotation speed, unit is steps/second. One full rotation of the servo is 4096 steps. The higher the value, the faster the speed. When the speed value is 0, it rotates at maximum speed.
acc: Acceleration at the start and end of rotation. The smaller the value, the smoother the start and stop. The value can be 0-254, with units of 100 steps/second^2. For example, if set to 10, the speed will vary according to an acceleration and deceleration of 1000 steps per second squared. When the acceleration value is 0, the rotation uses the maximum acceleration.

#### CMD_EOAT_GRAB_TORQUE
- {"T":107,"tor":200}
- Gripper force control
> The maximum value of tor can be 1000, representing 100% force.

#### CMD_SET_JOINT_PID
- {"T":108,"joint":3,"p":16,"i":0}
- Joint PID settings

#### CMD_RESET_PID
- {"T":109}
- Reset joint PID

#### CMD_SET_NEW_X
- {"T":110,"xAxisAngle":0}
- Set new X-axis direction

#### CMD_DYNAMIC_ADAPTATION
- {"T":112,"mode":0,"b":1000,"s":1000,"e":1000,"h":1000}
- Dynamic external force adaptive control

### M3 Robotic Arm Control

#### CMD_MOVE_INIT
- {"T":100}
- Move the robotic arm to its initial position
> Normally, the robotic arm will automatically move to its initial position when powered on. Using this command will block the process.

#### CMD_SINGLE_JOINT_CTRL
- {"T":101,"joint":0,"rad":0,"spd":0,"acc":10}
- Control the motion of a single joint of the robotic arm
> joint: The joint number.

1: BASE_JOINT - Base joint.

2: SHOULDER_JOINT - Shoulder joint.

3: ELBOW_JOINT - Elbow joint.

4: WRIST_JOINT - Wrist joint 1.

5: ROLL_JOINT - Wrist joint 2.

6: EOAT_JOINT - Gripper joint.

rad: The target rotation angle (in radians), relative to the joint's initial position. The default angles and rotation directions for each joint are as follows:

The BASE_JOINT starts at 0 radians, with a rotation range from 3.14 to -3.14 radians. Increasing the angle rotates the base joint to the left; decreasing the angle rotates it to the right.
The initial position of the SHOULDER_JOINT has a default angle of 0, with a rotation range from 1.57 to -1.57. Increasing the angle moves the shoulder joint forward, while decreasing the angle moves it backward.

The initial position of the ELBOW_JOINT has a default angle of 1.570796, with a rotation range from 3.14 to -1.11. Increasing the angle rotates the elbow joint downward, and decreasing the angle rotates it upward.

The initial position of the WRIST_JOINT has a default angle of 0, with a rotation range from 1.57 to -1.57. Increasing the angle rotates the wrist joint 1 downward, and decreasing the angle rotates it upward.

The initial position of the ROLL_JOINT has a default angle of 0, with a rotation range from 3.14 to -3.14. Increasing the angle rotates wrist joint 2 clockwise, while decreasing the angle rotates it counterclockwise.

The initial position of the EOAT_JOINT has a default angle of 3.141593, with a rotation range from 1.08 to 3.14. Decreasing the angle opens the gripper joint.
spd: Rotation speed, measured in steps per second. One complete rotation of the servo motor equals 4096 steps. A higher value means a faster speed. If the speed is set to 0, the joint rotates at maximum speed.
acc: The acceleration at the beginning and end of rotation. A smaller value produces a smoother start and stop. The value can range from 0 to 254, with units of 100 steps/sec². For example, setting it to 10 changes the speed with an acceleration of 1000 steps per second squared. If the acceleration is 0, movement occurs at maximum acceleration.

#### CMD_JOINTS_RAD_CTRL
- {"T":102,"base":0,"shoulder":0,"elbow":1.57,"wrist":0,"roll":0,"hand":1.57,"spd":0,"acc":10}
- Controls the rotation of all the robotic arm's joints in radians.
> base: Angle of the base joint. Rotation range is described in the "rad" key of the "CMD_SINGLE_JOINT_CTRL" command above.

shoulder: Angle of the shoulder joint.
elbow: Angle of the elbow joint.
wrist: Angle of the first wrist joint.
roll: Angle of the second wrist joint.

hand: Angle of the gripper joint.

spd: Rotational speed, in steps/sec. One full rotation of the servo equals 4096 steps. A higher value means faster movement. If set to 0, it moves at maximum speed.

acc: Acceleration at the start and end of rotation. A smaller value makes the start and stop smoother. The value can be 0-254, in units of 100 steps/sec². For example, if set to 10, the speed changes with an acceleration of 1000 steps per second squared. When set to 0, movement runs at maximum acceleration.

#### CMD_SINGLE_AXIS_CTRL
- {"T":103,"axis":2,"pos":0,"spd":0.25}
- Single-axis control
> The robotic arm's coordinate axes follow the right-hand rule: X-axis points forward, Y-axis points left from the front view, and Z-axis points vertically upward.

axis: Axis number. 1-X axis; 2-Y axis; 3-Z axis; 4-T axis (Pitch angle); 5-R axis (Roll angle); 6-G axis (Gripper angle), in radians.
pos: Specific position along an axis, in mm. For example, in the example above, the end-effector moves to position 0 on the Y-axis, i.e., directly in front of the arm.

spd: Movement speed. A higher value means faster movement. This movement command includes a curve speed control function, so the speed is not constant.

This command will block the process.

#### CMD_XYZT_GOAL_CTRL
- {"T":104,"x":235,"y":0,"z":234,"t":3.14,"spd":0.25}
- Robotic arm coordinate movement control (inverse kinematics control)
> x, y, z, t, r, g: Represent the positions of the six axes, in mm. For details, refer to the CMD_SINGLE_AXIS_CTRL command description above.

spd: Movement speed. A higher value means faster movement. This motion command includes a curve speed control function at the lower level, so the speed is not constant.

This command will block the process.

#### CMD_XYZT_DIRECT_CTRL
- {"T":1041,"x":235,"y":0,"z":234,"t":3.14}
- Robotic arm coordinate movement control (inverse kinematics control)
> x, y, z, t, r, g: Represent the positions of the six axes in millimeters. For details, see the CMD_SINGLE_AXIS_CTRL command description above.

Note: The difference between this command and the previous one is that this command does not cause the process to block. Since there is no interpolation calculation at the lower level, after this command is executed, the robotic arm will move to the target point at maximum speed. It is suitable for scenarios where new target points are continuously provided to the robot, and the target points of each command should not differ too much.

#### CMD_SERVO_RAD_FEEDBACK
- {"T":105}
- Feedback of the robotic arm's coordinate information

#### CMD_EOAT_HAND_CTRL
- {"T":106,"cmd":1.57,"spd":0,"acc":0}
- End effector joint control (in radians)
> cmd: The angle to rotate to (in radians). The initial position of EOAT_JOINT defaults to 3.141593 radians.
Gripper joint, angle rotation range: 1.08 to 3.14 radians. When the angle decreases, the gripper joint will open.

spd: Rotation speed, in steps per second. One servo rotation equals 4096 steps. The greater the value, the faster the speed. If spd is 0, it rotates at maximum speed.

acc: Acceleration at the start and end of rotation. The smaller the value, the smoother the start and stop. Values range from 0-254, units are 100 steps/second². For example, setting it to 10 means the speed changes according to an acceleration/deceleration of 1000 steps per second². When acceleration is 0, it runs at maximum acceleration.

#### CMD_EOAT_GRAB_TORQUE
- {"T":107,"tor":200}
- Gripper force control
> The maximum tor value is 1000, representing 100% force.

#### CMD_SET_JOINT_PID
- {"T":108,"joint":3,"p":16,"i":0}
- Joint PID settings
> joint indicates the joint number:

BASE_JOINT = 1

SHOULDER_JOINT = 2
ELBOW_JOINT = 3

WRIST_JOINT = 4

ROLL_JOINT = 5

GRIPPER = 6
The P value is the proportional coefficient, default is 16; too high a P value can cause the robotic arm joints to shake.

The I value is the integral coefficient, default is 0; can be set as a multiple of 8, used to compensate for position errors caused by payload, but too high an I value can cause vibration.

#### CMD_RESET_PID
- {"T":109}
- Reset joint PID

#### CMD_DYNAMIC_ADAPTATION
- {"T":112, "mode":0, "b":1000, "s":1000, "e":1000, "t":1000, "r":1000, "h":1000}
- Dynamic external force adaptive control

### Other Settings
#### CMD_HEART_BEAT_SET
- {"T":136,"cmd":3000}
- Set heartbeat function interval
> The 'cmd' unit is milliseconds. You can use this command to set the heartbeat function interval. If the lower machine does not receive a new movement command within this time, it will automatically stop moving. This is used to prevent the lower machine from continuing to move if the upper machine crashes during use.

#### CMD_SET_SPD_RATE
- {"T":138,"L":1,"R":1}
- Set left and right speed ratio
> The product uses differential steering. When the left and right wheels are given the same target speed, the product may not move straight due to encoder errors or tire traction differences. You can use this command to fine-tune the speed of the left and right wheels. For example, if the left wheel needs to move slower, you can set the value of L to 0.98. Try not to set L or R values greater than one.

#### CMD_GET_SPD_RATE
- {"T":139}
- Get current speed ratio
> This command allows you to get the current speed ratio.

### ESP-NOW Related Settings

#### CMD_BROADCAST_FOLLOWER
- {"T":300,"mode":1}
- {"T":300,"mode":0,"mac":"CC:DB:A7:5B:E4:1C"}
- Set the mode for ESP-NOW broadcast control
> When mode is 1, other devices can control through broadcast commands; when mode is 0, only the device with the specified MAC address can control.

#### CMD_GET_MAC_ADDRESS
- {"T":302}
- Get the MAC address of the current device

#### CMD_ESP_NOW_ADD_FOLLOWER
- {"T":303,"mac":"FF:FF:FF:FF:FF:FF"}
- Add a MAC address to the controlled device (PEER)

#### CMD_ESP_NOW_REMOVE_FOLLOWER
- {"T":304,"mac":"FF:FF:FF:FF:FF:FF"}
- Remove a MAC address from the PEER

#### CMD_ESP_NOW_GROUP_CTRL
- {"T":305,"dev":0,"b":0,"s":0,"e":1.57,"h":1.57,"cmd":0,"megs":"hello!"}
- ESP-NOW multicast control

#### CMD_ESP_NOW_SINGLE
- {"T":306,"mac":"FF:FF:FF:FF:FF:FF","dev":0,"b":0,"s":0,"e":1.57,"h":1.57,"cmd":0,"megs":"hello!"}
- ESP-NOW unicast/multicast control

### Task File Related Functions

This feature belongs to the advanced functions of the lower-level device. Normally, when used with the upper-level device, the following operations are usually not required.

#### CMD_SCAN_FILES
- {"T":200}
- Scan the current task files

#### CMD_CREATE_FILE
- {"T":201,"name":"file.txt","content":"inputContentHere."}
- Create a new task file

#### CMD_READ_FILE
- {"T":202,"name":"file.txt"}
- Read a task file

#### CMD_DELETE_FILE
- {"T":203,"name":"file.txt"}
- Delete a task file

#### CMD_APPEND_LINE
- {"T":204,"name":"file.txt","content":"inputContentHere."}
- Add a new instruction at the end of the task file

#### CMD_INSERT_LINE
- {"T":205,"name":"file.txt","lineNum":3,"content":"content"}
- Insert a new instruction in the middle of the task file

#### CMD_REPLACE_LINE
- {"T":206,"name":"file.txt","lineNum":3,"content":"Content"}
- Replace a certain instruction in the task file

### Bus Servo Settings

#### CMD_SET_SERVO_ID
- {"T":501,"raw":1,"new":11}
- Change servo ID
> 'raw' is the original ID of the servo (new servos are all 1), 'new' is the ID to change to, with a maximum of 254, cannot be negative, and 255 is the broadcast ID.

#### CMD_SET_MIDDLE
- {"T":502,"id":11}
- Set the current position of the servo as the servo midpoint (only valid for ST series servos).

#### CMD_SET_SERVO_PID
- {"T":503,"id":14,"p":16}
- Set the P value of the servo's PID.

### ESP32 Related Functions

#### CMD_REBOOT
- {"T":600}
- Reboot.

#### CMD_FREE_FLASH_SPACE
- {"T":601}
- Get the remaining FLASH space size.

#### CMD_BOOT_MISSION_INFO
- {"T":602}
- Output the current boot mission file.

#### CMD_RESET_BOOT_MISSION
- {"T":603}
- Reset the boot mission file.

#### CMD_NVS_CLEAR
- {"T":604}
- Clear the ESP32's NVS area. If Wi-Fi cannot connect successfully, you can try calling this command and then rebooting.

#### CMD_INFO_PRINT
- {"T":605,"cmd":1}
- Set the information feedback mode.
> When cmd is 1, print debug information; 2, continuous chassis information feedback; 0, no feedback.

#### CMD_MM_TYPE_SET
- {"T":900,"main":2,"module":0}
- Set the chassis type and chassis module type
> main:  
> 1 - RaspRover  
> 2 - UGV Rover  
> 3 - UGV Beast  
> module:  
> 0 - None  
> 1 - RoArm M2  
> 2 - Pan-Tilt  
> 3 - RoArm M3  